#   LinkedIn Post Generator


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import base64
import re
import html

# --- Global State ---
generated_post_content = ""

# =============================================================
# 1. SMART TEXT PROCESSING ENGINE
# =============================================================
def clean_text(text, make_title=False):
    text = text.strip()
    if not text:
        return ""
    if make_title:
        words = text.split()
        protected_words = []
        for w in words:
            if w.isupper() and len(w) > 1:
                protected_words.append(w)
            else:
                protected_words.append(w.capitalize())
        return " ".join(protected_words)
    sentences = re.split(r'(?<=[.!?])\s*', text)
    capitalized_sentences = [s[0].upper() + s[1:] for s in sentences if s.strip()]
    return " ".join(capitalized_sentences)


PROPER_NOUNS = {
    'ali', 'ahmed', 'hassan', 'fatima', 'sara', 'usman', 'zara', 'bilal', 'ayesha',
    'pakistan', 'karachi', 'lahore', 'islamabad', 'peshawar', 'quetta',
    'google', 'microsoft', 'amazon', 'meta', 'apple', 'netflix', 'openai',
    'python', 'pytorch', 'tensorflow', 'pandas', 'numpy', 'linkedin', 'github',
    'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday',
    'january', 'february', 'march', 'april', 'may', 'june',
    'july', 'august', 'september', 'october', 'november', 'december'
}

def capitalize_proper_nouns(text):
    """Capitalize known proper nouns anywhere in text."""
    def fix_word(w):
        clean = w.strip('.,!?:;()[]"\'')
        if clean.lower() in PROPER_NOUNS:
            return w.replace(clean, clean.capitalize(), 1)
        return w
    return ' '.join(fix_word(w) for w in text.split())


def get_article(word):
    """Extended — checks vowel sounds AND common vowel-sounding acronyms (L, M, N, R, S, X, H)."""
    if not word:
        return ""
    word_clean = word.strip().upper()
    vowel_sounds   = ['A', 'E', 'I', 'O', 'U']
    vowel_acronyms = ['L', 'M', 'N', 'R', 'S', 'X', 'H']
    if word_clean[0] in vowel_sounds or (len(word_clean) > 1 and word_clean[0] in vowel_acronyms):
        return "an"
    return "a"


def merge_and_format_tags(system_tags, user_tags_raw):
    """Dedup via set, sort alphabetically, capitalise first letter of each tag."""
    final_tags_set = set(tag.strip() for tag in system_tags)
    if user_tags_raw.strip():
        raw_splits = re.split(r'[\s,]+', user_tags_raw)
        for tag in raw_splits:
            clean_tag = tag.replace('#', '').strip()
            if clean_tag:
                formatted_user_tag = "#" + clean_tag[0].upper() + clean_tag[1:]
                final_tags_set.add(formatted_user_tag)
    return " ".join(sorted(list(final_tags_set)))


# =============================================================
# 2. TEMPLATE READABILITY — SEPARATE NAMED BLOCKS PER INTENT
#    Wrapped inside dual-language dict for language support
# =============================================================

# --- English (Professional) blocks ---
def _en_recruiter_internship(title, details, acc, tech, tags):
    return (
        f"🚨 New Opportunity Alert: '{title}'! 🚨\n\n"
        f"Hello network, our team is looking for talented interns. If you are passionate about "
        f"learning, driving impact, and working on modern systems, check out the parameters:\n\n"
        f"📌 Role Details & Requirements:\n{details}\n\n"
        f"Interested candidates can apply via link or drop CVs below. 👇\n\n{tags}"
    )

def _en_recruiter_job(title, details, acc, tech, tags):
    return (
        f"🚨 New Career Opportunity: Openings for '{title}'! 🚨\n\n"
        f"Hello network, my organization is looking for technical professionals. "
        f"Check out the details:\n\n"
        f"📌 Core Requirements:\n{details}\n\n"
        f"Apply via the application portal. Let's grow together! 👇\n\n{tags}"
    )

def _en_user_internship(title, details, acc, tech, tags):
    art = get_article(title)
    return (
        f"🎉 Excited to share a personal milestone! I have accepted an offer and am starting "
        f"{art} '{title}' role!\n\n"
        f"I am incredibly grateful for this opportunity to step into the workspace and contribute "
        f"to real-world pipelines.\n\n"
        f"💼 What I'll be focusing on:\n{details}\n\n"
        f"A huge thank you to everyone who supported me along the way! 🙏\n\n{tags}"
    )

def _en_user_job(title, details, acc, tech, tags):
    art = get_article(title)
    return (
        f"💼 Corporate Update: I'm thrilled to share that I'm starting a new position as "
        f"{art} '{title}'!\n\n"
        f"Looking forward to taking on new architectural challenges and collaborating with a brilliant team.\n\n"
        f"🚀 Core Responsibilities:\n{details}\n\n"
        f"Thank you to my network for the constant support! Let's build. ✨\n\n{tags}"
    )

def _en_project(title, details, acc, tech, tags):
    acc_line = f"• Global Accuracy Achieved: {acc}%\n" if acc else ""
    tech_line = tech if tech else "Modern Tech Stack"
    return (
        f"🚀 Building in Public: Scaled my latest project '{title}'!\n\n"
        f"I'm excited to share that I have successfully completed and optimized the "
        f"end-to-end processing script.\n\n"
        f"📊 Core Details & Insights:\n"
        f"{acc_line}{details}\n\n"
        f"💻 Tech Stack: {tech_line}.\n\n{tags}"
    )

def _en_learning(title, details, acc, tech, tags):
    return (
        f"📚 Today's Learning & Roadmap Share: '{title}'\n\n"
        f"Consistency is key in tech. Today, I focused on deep-diving into this domain "
        f"to strengthen my foundational grasp.\n\n"
        f"💡 Key Takeaways:\n{details}\n\n"
        f"What are you building today? Let's discuss in the comments! 👇\n\n{tags}"
    )

def _en_tips(title, details, acc, tech, tags):
    return (
        f"💡 Quick Tech Tip: {title}\n\n"
        f"Here is a quick workflow optimization hack that can save you significant debugging sprints:\n\n"
        f"🔍 Implementation & Logic:\n{details}\n\n"
        f"Hope this adds value to your framework! Save it for later usage. 📌\n\n{tags}"
    )


# --- Roman Urdu (Conversational) blocks ---
def _ur_recruiter_internship(title, details, acc, tech, tags):
    return (
        f"🚨 Naya Internship Alert: '{title}'! 🚨\n\n"
        f"Hello tech community, hamari team ko aise interns ki talaash hai jo seekhne aur "
        f"real-world systems par kaam karne ka jazba rakhte hon.\n\n"
        f"📌 Requirements:\n{details}\n\n"
        f"Agar aap interested hain toh niche diye gaye link par apply karein. 👇\n\n{tags}"
    )

def _ur_recruiter_job(title, details, acc, tech, tags):
    return (
        f"🚨 Career ka Naya Mauka: Openings for '{title}'! 🚨\n\n"
        f"Hello network, hamari organization mein ek behtareen role available hai. "
        f"Details niche check karein:\n\n"
        f"📌 Requirements:\n{details}\n\n"
        f"Interested log portal par apply kar sakte hain. Let's grow together! 👇\n\n{tags}"
    )

def _ur_user_internship(title, details, acc, tech, tags):
    return (
        f"🎉 Nayi shuruat! Mujhe share karte hue behad khushi ho rahi hai ke maine offer "
        f"accept kar li hai aur main baqaida '{title}' ke taur par start kar raha/rahi hoon!\n\n"
        f"Main is mauke ke liye bohot shukargzar hoon jahan mujhe industrial level ka exposure milega.\n\n"
        f"💼 Mera main focus kya hoga:\n{details}\n\n"
        f"Un sab logon ka shukriya jinhone is safar mein mera sath diya! 🙏\n\n{tags}"
    )

def _ur_user_job(title, details, acc, tech, tags):
    return (
        f"💼 Career Update: Mujhe yeh batate hue khushi ho rahi hai ke main ab naye position "
        f"'{title}' par apni nayi journey shuru kar raha/rahi hoon!\n\n"
        f"Naye challenges aur engineering optimizations ke liye fully excited hoon.\n\n"
        f"🚀 Mera Role aur Kaam:\n{details}\n\n"
        f"Mera sath karne ke liye pure network ka dil se shukriya! ✨\n\n{tags}"
    )

def _ur_project(title, details, acc, tech, tags):
    acc_line = f"• Hasil Kardah Accuracy: {acc}%\n" if acc else ""
    tech_line = tech if tech else "Python aur modern tools"
    return (
        f"🚀 Building in Public: Maine apna naya project '{title}' complete aur deploy kar liya hai!\n\n"
        f"Is architecture ki core processing ab flawlessly deploy ho chuki hai.\n\n"
        f"📊 Project ki Insights:\n"
        f"{acc_line}{details}\n\n"
        f"💻 Tech Stack: {tech_line}.\n\n{tags}"
    )

def _ur_learning(title, details, acc, tech, tags):
    return (
        f"📚 Aaj ki Learning aur Roadmap Update: '{title}'\n\n"
        f"Tech mein consistency hi sab kuch hai. Aaj maine is domain ke internal structure "
        f"ko deep-dive kiya.\n\n"
        f"💡 Asal Takeaways:\n{details}\n\n"
        f"Aap aaj kal kya seekh rahe hain? Niche comments mein discuss karte hain! 👇\n\n{tags}"
    )

def _ur_tips(title, details, acc, tech, tags):
    return (
        f"💡 Aik Choti Tech Tip: {title}\n\n"
        f"Yeh ek aisa quick workflow hack hai jo aapka development ka kafi time bacha sakta hai:\n\n"
        f"🔍 Kaise use karein aur Logic:\n{details}\n\n"
        f"Umeed hai yeh aapke daily sprint mein kaam aayega! Isay save karlein. 📌\n\n{tags}"
    )


# Dual-language dispatch dict
POST_TEMPLATES = {
    "English (Professional)": {
        "Recruiter - Internship Opportunity": _en_recruiter_internship,
        "Recruiter - Job Opportunity":        _en_recruiter_job,
        "User - Internship Offer":            _en_user_internship,
        "User - Job Offer":                   _en_user_job,
        "Project Upload":                     _en_project,
        "Daily Learning / Roadmap":           _en_learning,
        "Tips & Tricks":                      _en_tips,
    },
    "Roman Urdu (Conversational)": {
        "Recruiter - Internship Opportunity": _ur_recruiter_internship,
        "Recruiter - Job Opportunity":        _ur_recruiter_job,
        "User - Internship Offer":            _ur_user_internship,
        "User - Job Offer":                   _ur_user_job,
        "Project Upload":                     _ur_project,
        "Daily Learning / Roadmap":           _ur_learning,
        "Tips & Tricks":                      _ur_tips,
    }
}

BASE_TAGS_MAP = {
    "Recruiter - Internship Opportunity": ["#Hiring", "#InternshipOpportunity", "#TechInterns", "#Recruitment", "#DataScience"],
    "Recruiter - Job Opportunity":        ["#Hiring", "#JobOpportunity", "#TechJobs", "#CareerOpenings", "#Recruitment"],
    "User - Internship Offer":            ["#Internship", "#CareerUpdate", "#NewJourney", "#GrowthMindset"],
    "User - Job Offer":                   ["#NewJob", "#CareerGrowth", "#TechIndustry", "#Engineering", "#Jobs"],
    "Project Upload":                     ["#MachineLearning", "#DataScience", "#BuildingInPublic", "#TechInnovation"],
    "Daily Learning / Roadmap":           ["#ContinuousLearning", "#TechRoadmap", "#Python", "#TechCommunity"],
    "Tips & Tricks":                      ["#TechTips", "#PythonHacks", "#CodingLife", "#Productivity"],
}


# =============================================================
# 3. HYBRID INTENT CLASSIFIER
# =============================================================
def determine_final_intent(selected_dropdown, title, details):
    combined_text = f"{title.lower()} {details.lower()}"

    recruiter_triggers = ['hiring', 'apply now', 'opening', 'vacancy', 'opportunities',
                          'looking for', 'deadline', 'we are hiring', 'join our team']
    candidate_triggers = ['accepted', 'started', 'joined', 'thrilled', 'happy to share',
                          'milestone', 'offer', 'excited to share']  # kept for future use

    if selected_dropdown == "Internship Offer":
        if any(kw in combined_text for kw in recruiter_triggers):
            return "Recruiter - Internship Opportunity"
        return "User - Internship Offer"

    elif selected_dropdown == "Job Offer":
        if any(kw in combined_text for kw in recruiter_triggers):
            return "Recruiter - Job Opportunity"
        return "User - Job Offer"

    elif selected_dropdown == "Hiring / Opportunity Share":
        if any(kw in combined_text for kw in ['intern', 'internship']):
            return "Recruiter - Internship Opportunity"
        return "Recruiter - Job Opportunity"

    elif selected_dropdown == "Project Upload":
        if any(kw in combined_text for kw in ['tip', 'trick', 'hack', 'shortcut']):
            return "Tips & Tricks"
        return "Project Upload"

    return selected_dropdown  # "Daily Learning / Roadmap" or "Tips & Tricks" pass-through


# =============================================================
# 4. GENERATE POST — MAIN HANDLER
# =============================================================
def generate_post(b):
    global generated_post_content
    with output_area:
        clear_output()

        selected_lang = language_dropdown.value
        dropdown_val  = category_dropdown.value
        title         = clean_text(title_input.value, make_title=True)
        tech_stack    = clean_text(tech_input.value,  make_title=True)
        details       = capitalize_proper_nouns(clean_text(details_input.value, make_title=False))
        custom_tags   = tags_input.value.strip()

        acc_raw      = accuracy_input.value.strip()
        acc_filtered = re.sub(r'[^0-9.%]', '', acc_raw)
        acc_clean    = acc_filtered.rstrip('%').strip() if acc_filtered else ""

        if not title and not details:
            print("⚠️  Error: Please fill in Title or Details content first!")
            return

        final_intent = determine_final_intent(dropdown_val, title, details)
        merged_tags  = merge_and_format_tags(BASE_TAGS_MAP[final_intent], custom_tags)

        generated_post_content = POST_TEMPLATES[selected_lang][final_intent](
            title, details, acc_clean, tech_stack, merged_tags
        )

        print("\n================ ✨ HYBRID GENERATED POST ✨ ================\n")
        print(generated_post_content)
        print("\n=============================================================")

        display(action_button_layout)

        # --- File Attachments ---
        if file_upload.value:
            files_list = []
            if isinstance(file_upload.value, dict):
                for f_name, f_info in file_upload.value.items():
                    files_list.append({'name': f_name, 'content': f_info['content']})
            elif isinstance(file_upload.value, (tuple, list)):
                files_list = list(file_upload.value)

            print(f"\n📂 Attached Files ({len(files_list)}):")
            image_widgets = []

            for file_item in files_list:
                f_name    = file_item['name']
                f_content = file_item['content']
                safe_name = html.escape(f_name)
                b64       = base64.b64encode(f_content).decode()

                href = (
                    f'<div style="margin:4px 0;">'
                    f'<a href="data:application/octet-stream;base64,{b64}" '
                    f'download="{safe_name}" '
                    f'style="color:#1a73e8; font-weight:bold; text-decoration:underline;">'
                    f'👉 Download Asset: {safe_name}</a></div>'
                )
                display(HTML(href))

                if f_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                    img_w = widgets.Image(
                        value=f_content, format='png', width=150,
                        layout=widgets.Layout(margin='5px', border='1px solid #ddd')
                    )
                    image_widgets.append(img_w)

            if image_widgets:
                display(widgets.HBox(
                    children=image_widgets,
                    layout=widgets.Layout(flex_flow='row wrap')
                ))


# =============================================================
# 5. UTILITY ACTIONS
# =============================================================
def copy_to_clipboard(b):
    escaped = html.escape(generated_post_content).replace('\n', '\\n').replace("'", "\\'")
    js_code = f"""
    <script>
    navigator.clipboard.writeText('{escaped}').then(() => {{
        alert('📋 Post copied to clipboard!');
    }}).catch(err => {{
        alert('❌ Copy failed: ' + err);
    }});
    </script>
    """
    display(HTML(js_code))

def redirect_to_linkedin(b):
    display(HTML("<script>window.open('https://www.linkedin.com/feed/', '_blank');</script>"))

def on_file_upload(change):
    with upload_status_area:
        clear_output()
        if change['new']:
            files = change['new']
            names = list(files.keys()) if isinstance(files, dict) else [f['name'] for f in files]
            if len(names) == 1:
                print(f"✅ File Loaded Successfully: {names[0]}")
            else:
                print(f"✅ {len(names)} Files Loaded Successfully: {', '.join(names)}")

def on_category_change(change):
    c = change['new']
    if c == "Project Upload":
        accuracy_input.layout.display = 'flex'
        tech_input.layout.display     = 'flex'
    elif c in ["Internship Offer", "Job Offer", "Hiring / Opportunity Share"]:
        accuracy_input.layout.display = 'none'
        tech_input.layout.display     = 'none'
    else:
        accuracy_input.layout.display = 'none'
        tech_input.layout.display     = 'flex'


# =============================================================
# 6. UI LAYOUT
# =============================================================
language_dropdown = widgets.Dropdown(
    options=['English (Professional)', 'Roman Urdu (Conversational)'],
    value='English (Professional)',
    description='Post Language:',
    style={'description_width': 'initial'}
)

category_dropdown = widgets.Dropdown(
    options=['Project Upload', 'Internship Offer', 'Job Offer',
             'Hiring / Opportunity Share', 'Daily Learning / Roadmap', 'Tips & Tricks'],
    value='Project Upload',
    description='Post Type:',
    style={'description_width': 'initial'}
)

title_input = widgets.Text(
    value='', placeholder='e.g., L-shaped Curve Analysis',
    description='Title / Role:', style={'description_width': 'initial'}
)

tech_input = widgets.Text(
    value='', placeholder='e.g., Python, Pandas',
    description='Tech Stack:', style={'description_width': 'initial'}
)

accuracy_input = widgets.Text(
    value='', placeholder='e.g., 94.2',
    description='Accuracy (%):', style={'description_width': 'initial'}
)

tags_input = widgets.Text(
    value='',
    placeholder='e.g., career, software, openSource (comma or space separated)',
    description='Custom Tags:',
    style={'description_width': 'initial'}
)

details_input = widgets.Textarea(
    value='',
    placeholder='Post ka details/context likhein.',
    description='Main Context:',
    rows=8,
    layout=widgets.Layout(width='95%', height='auto')
)

file_upload = widgets.FileUpload(
    accept='', multiple=True,
    description='Attach Files', button_style='info', icon='paperclip'
)
file_upload.observe(on_file_upload, names='value')

generate_btn = widgets.Button(
    description="⚡ Generate  Post",
    button_style='success', icon='bolt',
    layout=widgets.Layout(width='95%', margin='10px 0px')
)
generate_btn.on_click(generate_post)

btn_copy     = widgets.Button(description="📋 Copy Post",          button_style='primary')
btn_linkedin = widgets.Button(description="🌐 Open LinkedIn Feed", button_style='warning')
btn_copy.on_click(copy_to_clipboard)
btn_linkedin.on_click(redirect_to_linkedin)
action_button_layout = widgets.HBox(
    [btn_copy, btn_linkedin],
    layout=widgets.Layout(margin='15px 0px')
)

upload_status_area = widgets.Output()
output_area        = widgets.Output()

category_dropdown.observe(on_category_change, names='value')

# =============================================================
# 7. RENDER UI
# =============================================================
print("👑 Hybrid LinkedIn Post Generator — All Features Active:")
display(
    language_dropdown,
    category_dropdown,
    title_input,
    tech_input,
    accuracy_input,
    details_input,
    tags_input,
    file_upload,
    upload_status_area,
    generate_btn,
    output_area
)


👑 Hybrid LinkedIn Post Generator — All Features Active:


Dropdown(description='Post Language:', options=('English (Professional)', 'Roman Urdu (Conversational)'), styl…

Dropdown(description='Post Type:', options=('Project Upload', 'Internship Offer', 'Job Offer', 'Hiring / Oppor…

Text(value='', description='Title / Role:', placeholder='e.g., L-shaped Curve Analysis', style=TextStyle(descr…

Text(value='', description='Tech Stack:', placeholder='e.g., Python, Pandas', style=TextStyle(description_widt…

...

Text(value='', description='Custom Tags:', placeholder='e.g., career, software, openSource (comma or space sep…

FileUpload(value=(), button_style='info', description='Attach Files', icon='paperclip', multiple=True)

Output()

Button(button_style='success', description='⚡ Generate  Post', icon='bolt', layout=Layout(margin='10px 0px', w…

Output()